<a href="https://colab.research.google.com/github/Learn-AIMLOps/Instruction_fine_tunning/blob/main/ChatML_finetune_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# STEP 1 — Install Libraries

!pip install -q transformers accelerate peft bitsandbytes trl datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 23.3 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
hf_write_token = userdata.get("HF_WRITE_TOKEN")

In [3]:
# STEP 2 — Login to HuggingFace
from huggingface_hub import login
login(token=hf_write_token)

In [4]:
# STEP 3 — Load Base Model (4bit)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [5]:
# STEP 4 — Load Domain Adapter

from peft import PeftModel

domain_adapter_path = "RohitGu/Qwen2-1.5B-LLMOps-LoRA"

model_with_domain = PeftModel.from_pretrained(
    base_model,
    domain_adapter_path
)

adapter_config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

In [6]:
# STEP 5 — Merge Domain Adapter

merged_model = model_with_domain.merge_and_unload()

merged_model.save_pretrained("merged_domain_model")
tokenizer.save_pretrained("merged_domain_model")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('merged_domain_model/tokenizer_config.json',
 'merged_domain_model/chat_template.jinja',
 'merged_domain_model/tokenizer.json')

In [7]:
# STEP 6 — Reload merged model in 4bit

base_model = AutoModelForCausalLM.from_pretrained(
    "merged_domain_model",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
# STEP 7 — Load Stage 4 Instruction Adapter

stage4_adapter = "RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA"

model = PeftModel.from_pretrained(
    base_model,
    stage4_adapter
)

adapter_config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

In [10]:
# STEP 8 — Pre Training Test
# It may respond but not perfectly role-aware.

def generate_response(model, tokenizer, messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Explain LLM in simple terms."}
]

print(generate_response(model, tokenizer, test_messages))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


system
You are helpful.
user
Explain LLM in simple terms.
assistant
LLMs (Language Model) are advanced artificial intelligence systems that can understand and generate human-like text. They are trained on vast amounts of text data, including books, articles, and web pages, to learn how to generate coherent and contextually relevant text. LLMs are particularly useful for tasks such as text summarization, question answering, and text generation. They are also used in natural language processing (NLP) applications such as language translation, speech recognition, and sentiment analysis. LLMs are a powerful tool for automating tasks that require human-level language understanding and generation. They are particularly useful for tasks that require a high degree of accuracy and consistency, such as legal document translation, medical record transcription, and financial reporting. LLMs are also being used in areas such as cybersecurity, fraud detection, and fraud prevention. They are particul

In [11]:
# STEP 9 — Load Stage-4 Alpaca Dataset
from datasets import load_dataset

dataset = load_dataset("RohitGu/llmops-guide-ift-datasets")

README.md:   0%|          | 0.00/344 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/30.0k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/91 [00:00<?, ? examples/s]

In [13]:
# Check Your Dataset Schema
print(dataset["train"][0])
print(dataset["train"].features)

{'instruction': 'Explain why small language models offer a more resource-efficient alternative than larger ones.', 'input': 'Small language models are often used in conjunction with large-scale pre-trained models to scale up the computational resources required to train them.', 'response': 'Small language models require less computational power and data to train compared to larger models, making them a more resource-efficient option.'}
{'instruction': Value('string'), 'input': Value('string'), 'response': Value('string')}


In [17]:
# STEP 10 — Convert to ChatML

def convert_to_chatml(example):
    system_prompt = "You are a helpful AI assistant."

    user_content = example["instruction"]
    if example.get("input"):
        user_content += "\n" + example["input"]

    chatml_text = (
        "<|im_start|>system\n"
        f"{system_prompt}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_content}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['response']}\n"          # ← changed from 'output' to 'response'
        "<|im_end|>"
    )

    return {"text": chatml_text}


# Apply the mapping
chat_dataset = dataset["train"].map(convert_to_chatml)


# Then apply it
chat_dataset = dataset["train"].map(convert_to_chatml)

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

In [18]:
# STEP 11 — Save Chat Dataset to HF
chat_dataset.push_to_hub("RohitGu/llmops-guide-chatml-dataset", token=hf_write_token)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 58.4kB / 58.4kB            

CommitInfo(commit_url='https://huggingface.co/datasets/RohitGu/llmops-guide-chatml-dataset/commit/c8112b7ea8cbba32e1e25474f2677d8a6a765721', commit_message='Upload dataset', commit_description='', oid='c8112b7ea8cbba32e1e25474f2677d8a6a765721', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/RohitGu/llmops-guide-chatml-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='RohitGu/llmops-guide-chatml-dataset'), pr_revision=None, pr_num=None)

In [24]:
# STEP 12 — Stable TRL Chat Alignment Training (Colab T4 Safe)

from trl import SFTTrainer, SFTConfig

# Enable memory optimization
model.gradient_checkpointing_enable()

sft_config = SFTConfig(
    output_dir="./chat_adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    packing=False
)

# Formatting function (Required in latest TRL)
def formatting_func(example):
    return example["text"]


trainer = SFTTrainer(
    model=model,
    train_dataset=chat_dataset,
    formatting_func=formatting_func,
    args=sft_config
)

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/91 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.321423
20,2.331994


TrainOutput(global_step=24, training_loss=2.3080290158589682, metrics={'train_runtime': 166.6674, 'train_samples_per_second': 1.092, 'train_steps_per_second': 0.144, 'total_flos': 167834516520960.0, 'train_loss': 2.3080290158589682})

In [25]:
# STEP 13 — Save Chat Adapter
trainer.model.save_pretrained("chat_adapter")
tokenizer.save_pretrained("chat_adapter")

('chat_adapter/tokenizer_config.json',
 'chat_adapter/chat_template.jinja',
 'chat_adapter/tokenizer.json')

In [26]:
# STEP 14 — Push Chat Adapter to HF
trainer.model.push_to_hub("RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA", token=hf_write_token)
tokenizer.push_to_hub("RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA", token=hf_write_token)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  14%|#4        | 1.23MB / 8.73MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpe63xul1m/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA/commit/7b5c604ca22c2acc9f6b121e3dc2307618f123d5', commit_message='Upload tokenizer', commit_description='', oid='7b5c604ca22c2acc9f6b121e3dc2307618f123d5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA', endpoint='https://huggingface.co', repo_type='model', repo_id='RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA'), pr_revision=None, pr_num=None)

In [27]:
# STEP 15 — Post Training Test

chat_model = PeftModel.from_pretrained(
    base_model,
    "RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA"
)

print(generate_response(chat_model, tokenizer, test_messages))

adapter_config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/8.73M [00:00<?, ?B/s]

system
You are helpful.
user
Explain LLM in simple terms.
assistant
LLMs (Language Model) are advanced artificial intelligence systems that can understand and generate human-like text. They are trained on vast amounts of text data, including books, articles, and web pages, to learn how to generate coherent and contextually relevant text. LLMs are particularly useful for tasks such as text summarization, question answering, and text generation. They are also used in natural language processing (NLP) applications such as language translation, speech recognition, and sentiment analysis. LLMs are a powerful tool for automating tasks that require human-level language understanding and generation. They are particularly useful for tasks that require a high degree of accuracy and consistency, such as legal document translation, medical record transcription, and financial reporting. LLMs are also being used in areas such as cybersecurity, fraud detection, and fraud prevention. They are particul